# 🧠 Introducción a RAG — Retrieval Augmented Generation
### Versión gratuita — Sin API keys 🔓
### Materia: Procesamiento de Lenguaje Natural — Nivel Terciario

---

En este notebook vamos a recorrer el camino que va desde entender **por qué los LLMs tienen limitaciones estructurales** hasta comprender cómo RAG las resuelve.

**✨ Novedad de esta versión:** Todo corre con herramientas **100% gratuitas y open-source**. No necesitás API key de ningún servicio. Usamos modelos de Hugging Face 🤗 que se descargan automáticamente.

Al final de este notebook vas a poder responder:
- ¿Por qué un LLM puede "mentir" con total confianza?
- ¿Cómo genera texto un modelo de lenguaje por dentro?
- ¿Qué es RAG y cómo resuelve esas limitaciones?
- ¿Qué alternativas y evoluciones existen hoy?

## ⚙️ ¿Qué herramientas vamos a usar?

| Componente | Herramienta gratuita | Reemplaza a |
|---|---|---|
| **LLM generador** | Qwen2.5-1.5B-Instruct (Hugging Face) | GPT-4o-mini (OpenAI, pago) |
| **Embeddings** | Sentence Transformers (multilingual) | text-embedding-3-small (OpenAI, pago) |
| **Vector Store** | ChromaDB (open-source) | ChromaDB (igual, ya es gratis) |
| **Infraestructura** | Google Colab (GPU T4 gratis) | Servidores externos |

> 💡 Todo corre localmente en la máquina de Colab. **Cero llamadas a APIs externas, cero costos.**

In [1]:
# 📦 Instalación de dependencias gratuitas
# Transformers + Sentence Transformers + ChromaDB + bitsandbytes (para cuantización 4-bit)

!pip install transformers accelerate sentence-transformers chromadb bitsandbytes torch numpy --quiet

print('✅ Dependencias instaladas correctamente')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [2]:
# 🔧 Importaciones y configuración del dispositivo

import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

# Detectar GPU disponible
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Dispositivo detectado: {device.upper()}")

if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memoria: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("   ⚠️  No se detectó GPU. Usaremos CPU (más lento pero funciona igual).")
    print("   💡 En Colab, activá la GPU en: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU")

🖥️  Dispositivo detectado: CPU
   ⚠️  No se detectó GPU. Usaremos CPU (más lento pero funciona igual).
   💡 En Colab, activá la GPU en: Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU


In [3]:
# 🤖 Cargamos el modelo de lenguaje gratuito (Qwen2.5-1.5B-Instruct)
# Es un modelo chico pero potente, con excelente soporte para español.
# La descarga inicial toma ~30-60 segundos. Las ejecuciones siguientes usan caché.

print("📥 Descargando modelo (solo la primera vez)...")
print("   Modelo: Qwen/Qwen2.5-1.5B-Instruct")
print("   Tamaño: ~3 GB (comprimido a 4-bit para ahorrar memoria)\n")

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

try:
    # Intentamos cargar con cuantización 4-bit (ahorra mucha memoria GPU)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        load_in_4bit=True,
        torch_dtype=torch.float16,
        trust_remote_code=True
    )
    modo_carga = "4-bit (GPU optimizada)"
except Exception:
    # Fallback: carga normal (CPU o GPU sin cuantización)
    print("   ⚠️  No se pudo cargar en 4-bit. Usando modo estándar...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        trust_remote_code=True
    )
    modo_carga = "estándar"

print(f"✅ Modelo cargado exitosamente ({modo_carga})")
print(f"   Parámetros: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

📥 Descargando modelo (solo la primera vez)...
   Modelo: Qwen/Qwen2.5-1.5B-Instruct
   Tamaño: ~3 GB (comprimido a 4-bit para ahorrar memoria)



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

   ⚠️  No se pudo cargar en 4-bit. Usando modo estándar...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Modelo cargado exitosamente (estándar)
   Parámetros: 1.5B


In [4]:
# Función auxiliar para generar texto con nuestro modelo local

def generar_texto(
    prompt: str,
    temperature: float = 0.7,
    max_new_tokens: int = 200,
    verbose: bool = False
) -> str:
    """Genera texto usando el modelo local Qwen2.5"""

    messages = [{"role": "user", "content": prompt}]

    # Aplicar el chat template del modelo
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            top_p=0.95 if temperature > 0 else 1.0,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decodificar solo los tokens generados (sin el prompt)
    response = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:],
        skip_special_tokens=True
    )

    if verbose:
        print(f"🌡️  Temperature: {temperature}")
        print(f"📏 Tokens generados: {len(outputs[0]) - inputs.input_ids.shape[1]}")

    return response

---
# 📌 Parte 1 — ¿Por qué los LLMs mienten con tanta confianza?

Un LLM fue entrenado con texto de internet hasta cierta fecha. Eso implica tres limitaciones estructurales:

| Problema | Descripción | Ejemplo |
|---|---|---|
| **Knowledge cutoff** | No sabe nada posterior a su fecha de entrenamiento | "¿Quién ganó las elecciones de 2025?" |
| **Alucinación** | Genera texto plausible aunque sea falso | Inventa citas, leyes, personas |
| **Conocimiento privado** | No tiene acceso a documentos internos de tu empresa | Manuales, políticas, stock |

Veamos el problema en acción 👇

In [5]:
# 🔴 DEMO: El LLM sin contexto — observá la respuesta
# Preguntamos sobre una empresa ficticia que el modelo NO conoce

pregunta = """¿Cuál es la política de devoluciones de la empresa TechStore Argentina
para productos electrónicos comprados online?"""

print("❓ Pregunta:", pregunta)
print("\n🤖 Respuesta del LLM sin contexto:")
print("-" * 60)

respuesta = generar_texto(pregunta, temperature=0.3)
print(respuesta)
print("-" * 60)
print("\n⚠️  El modelo respondió con confianza, pero ¿de dónde sacó esa información?")
print("    Simplemente generó texto plausible. Eso es alucinación.")
print("    No conoce TechStore Argentina, pero igual 'inventó' una respuesta.")

❓ Pregunta: ¿Cuál es la política de devoluciones de la empresa TechStore Argentina
para productos electrónicos comprados online?

🤖 Respuesta del LLM sin contexto:
------------------------------------------------------------
Lo siento por cualquier confusión, pero como asistente de inteligencia artificial, no tengo acceso directo a información sobre políticas de devolución específicas de empresas en particular. Te recomendaría que consultes directamente la página web oficial de TechStore Argentina o contacte su soporte al cliente para obtener más detalles y instrucciones sobre las políticas de devoluciones relacionadas con los productos electrónicos que han sido adquiridos online.
------------------------------------------------------------

⚠️  El modelo respondió con confianza, pero ¿de dónde sacó esa información?
    Simplemente generó texto plausible. Eso es alucinación.
    No conoce TechStore Argentina, pero igual 'inventó' una respuesta.


### 🔍 ¿Qué observás en la respuesta anterior?

El modelo respondió con total confianza sobre una empresa ficticia. No dijo "no sé", inventó una política que suena razonable. **Esto es alucinación.**

Esto no es un bug, es una consecuencia directa de **cómo funciona el modelo por dentro**. Para entenderlo hay que abrir la caja negra.

---
# 📌 Parte 2 — Cómo funciona un LLM por dentro

## 2.1 — Softmax: el corazón matemático

Antes de ver el modelo completo, entendamos la **función Softmax**, que convierte números crudos (logits) en probabilidades que suman 1.

```
Softmax(x_i) = exp(x_i) / SUM(exp(x_j) para todo j)
```

El parámetro **temperature** escala los logits *antes* del Softmax:

```
Softmax(x_i / temperature)
```

| Temperature | Efecto |
|---|---|
| `0` (greedy) | Siempre el token más probable |
| `0.2 - 0.5` | Distribución afilada, poco azar |
| `0.7 - 1.0` | Distribución balanceada |
| `1.5+` | Distribución muy plana, mucho azar |

Veamos el efecto matemático 👇

In [6]:
# 🧮 DEMO: Softmax con diferentes temperatures (puro numpy, sin modelo)
# Simulamos logits para 5 palabras candidatas

def softmax(logits, temperature=1.0):
    """Calcula softmax con temperature"""
    if temperature == 0:
        # Temperature 0 = greedy: una probabilidad 1, el resto 0
        probs = np.zeros_like(logits)
        probs[np.argmax(logits)] = 1.0
        return probs
    scaled = logits / temperature
    exp = np.exp(scaled - np.max(scaled))  # resta por estabilidad numérica
    return exp / exp.sum()

# Simulamos logits como los que generaría un Transformer
palabras = ["el", "cielo", "es", "azul", "hermoso"]
logits = np.array([1.2, 0.8, 0.5, 3.5, 1.8])  # "azul" tiene el logit más alto

temperatures = [0, 0.3, 1.0, 2.0]

print("📊 Logits originales:", logits)
print("   Palabras:", palabras)
print("\n" + "=" * 60)

for temp in temperatures:
    probs = softmax(logits, temp)
    palabra_elegida = palabras[np.argmax(probs)]
    print(f"\n🌡️  Temperature = {temp}")
    print(f"    {'  '.join([f'{p}: {prob:.3f}' for p, prob in zip(palabras, probs)])}")
    print(f"    ➡️  Token elegido: '{palabra_elegida}'")

print("\n" + "=" * 60)
print("💡 Con temp baja → siempre 'azul' (el más probable).")
print("   Con temp alta → se mezclan las probabilidades, más variedad.")

📊 Logits originales: [1.2 0.8 0.5 3.5 1.8]
   Palabras: ['el', 'cielo', 'es', 'azul', 'hermoso']


🌡️  Temperature = 0
    el: 0.000  cielo: 0.000  es: 0.000  azul: 1.000  hermoso: 0.000
    ➡️  Token elegido: 'azul'

🌡️  Temperature = 0.3
    el: 0.000  cielo: 0.000  es: 0.000  azul: 0.996  hermoso: 0.003
    ➡️  Token elegido: 'azul'

🌡️  Temperature = 1.0
    el: 0.072  cielo: 0.048  es: 0.036  azul: 0.714  hermoso: 0.130
    ➡️  Token elegido: 'azul'

🌡️  Temperature = 2.0
    el: 0.142  cielo: 0.116  es: 0.100  azul: 0.449  hermoso: 0.192
    ➡️  Token elegido: 'azul'

💡 Con temp baja → siempre 'azul' (el más probable).
   Con temp alta → se mezclan las probabilidades, más variedad.


## 2.2 — El loop autoregresivo en acción

Ahora veamos cómo el modelo real genera texto paso a paso.
El mecanismo central del LLM es:

```
Dado el texto que vino antes → ¿cuál es el token más probable que sigue?
```

Genera un token, lo agrega a la entrada, genera el siguiente... así hasta completar la respuesta.

In [7]:
# 🎛️ DEMO: El efecto de la temperature en el modelo real

prompt = "Continuá esta oración de forma creativa: 'El vendedor miró el inventario y...'"

temperatures = [0.0, 0.7, 1.5]

print(f"📝 Prompt: {prompt}\n")
print("=" * 60)

for temp in temperatures:
    print(f"\n🌡️  Temperature = {temp}")
    print("-" * 40)
    # Generamos 2 veces para mostrar la variabilidad
    for i in range(2):
        resultado = generar_texto(prompt, temperature=temp, max_new_tokens=60)
        print(f"  Intento {i+1}: {resultado}")

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


📝 Prompt: Continuá esta oración de forma creativa: 'El vendedor miró el inventario y...'


🌡️  Temperature = 0.0
----------------------------------------
  Intento 1: 'El vendedor miró el inventario y decidió que era hora de hacer una pausa para tomar un café fresco en la terraza del establecimiento.'
  Intento 2: 'El vendedor miró el inventario y decidió que era hora de hacer una pausa para tomar un café fresco en la terraza del establecimiento.'

🌡️  Temperature = 0.7
----------------------------------------
  Intento 1: "El vendedor miró el inventario con una mezcla de ansiedad y emoción, sabiendo que cada producto tenía la capacidad de cambiar su vida."
  Intento 2: 'El vendedor miró el inventario y sintió un escalofrío recorrer su espalda. ¿Cómo era posible que en este mundo tan moderno y avispado, hubiera algo aún por descubrir? Sus ojos se detuvieron en la última

🌡️  Temperature = 1.5
----------------------------------------
  Intento 1: "Estaba seguros de que su almacén estaba

### 🔍 ¿Qué observás?

- **Temperature 0**: las respuestas son casi idénticas. El modelo siempre elige el token más probable.
- **Temperature 0.7**: hay variación pero las respuestas siguen siendo coherentes.
- **Temperature 1.5**: las respuestas pueden ser muy distintas, más creativas o incluso incoherentes.

> 💡 **Esto explica algo importante sobre RAG:** el mismo sistema RAG puede dar respuestas distintas a la misma pregunta. No es un bug, es la naturaleza probabilística del generador.

In [8]:
# 🔬 EXPERIMENTO LIBRE: probá distintos prompts y temperatures
# Modificá las variables abajo y ejecutá la celda

mi_prompt = "Explicá en 3 oraciones qué es el aprendizaje automático."
mis_temperatures = [0.0, 1.0]

for temp in mis_temperatures:
    print(f"\n" + "=" * 50)
    resultado = generar_texto(mi_prompt, temperature=temp, verbose=True)
    print(f"💬 Respuesta:\n{resultado}")


🌡️  Temperature: 0.0
📏 Tokens generados: 199
💬 Respuesta:
El aprendizaje automático es un campo de estudio que se ocupa de enseñar a los sistemas computacionales a aprender y mejorar su rendimiento sin ser explicitamente programados para hacerlo. Esto implica la capacidad de los algoritmos y modelos de aprendizaje automático para analizar datos, identificar patrones y relaciones entre ellos, y utilizar esa información para tomar decisiones o realizar predicciones.

En otras palabras, el aprendizaje automático permite a las máquinas "aprender" de grandes cantidades de datos y aplicar ese conocimiento para resolver problemas complejos y predecir resultados con base en nuevos datos no previamente vistas. Este proceso involucra técnicas como la regresión, la clasificación, la clustering, la redes neuronales y otros métodos avanzados del aprendizaje profundo, lo que hace posible una amplia gama de aplicaciones en diversas industrias y campos de investigación.

🌡️  Temperature: 1.0
📏 Tokens

---
# 📌 Parte 3 — El pipeline RAG completo

Ahora que entendemos las limitaciones del LLM y cómo funciona por dentro, RAG tiene sentido como solución.

**La idea central de RAG:**

> En vez de esperar que el modelo "sepa" la respuesta, le *damos* la información relevante en el prompt.

## El pipeline en dos etapas

```
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ETAPA 1 — INDEXACIÓN (se hace una vez, offline)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Documentos → Chunking → Embeddings → Vector Store
  📄📄📄         ✂️✂️✂️       🔢🔢🔢        🗄️

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ETAPA 2 — RECUPERACIÓN Y GENERACIÓN (en tiempo real)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Pregunta → Embedding → Similitud → Top-K chunks
                                        ↓
                            Prompt aumentado
                            = instrucción + contexto + pregunta
                                        ↓
                                      LLM
                                        ↓
                                    Respuesta
```

**Diferencia clave con la versión original:**
- Embeddings: usamos **Sentence Transformers** en vez de OpenAI
- LLM generador: usamos **Qwen2.5** en vez de GPT
- Costo: **$0** 🎉

In [9]:
# 📦 Instalamos las librerías del pipeline RAG (si no lo hiciste antes)
!pip install chromadb sentence-transformers --quiet
print('✅ ChromaDB y sentence-transformers listos')

✅ ChromaDB y sentence-transformers listos


In [10]:
# 📄 CORPUS DE EJEMPLO — Política de empresa ficticia
# En los Colabs siguientes vas a poder reemplazar esto con tus propios documentos

documentos = [
    """POLÍTICA DE DEVOLUCIONES — TechStore Argentina
    Los clientes pueden devolver productos electrónicos dentro de los 30 días corridos
    desde la fecha de compra, siempre que presenten el ticket original y el producto
    esté en su embalaje original sin señales de uso.""",

    """GARANTÍA DE PRODUCTOS — TechStore Argentina
    Todos los productos tienen garantía legal de 6 meses por defectos de fabricación.
    Los televisores y notebooks tienen garantía extendida de 12 meses.
    La garantía no cubre daños por mal uso o accidentes.""",

    """POLÍTICA DE ENVÍOS — TechStore Argentina
    Los envíos al interior del país tienen un plazo de 3 a 7 días hábiles.
    El envío es gratuito para compras superiores a $50.000.
    Para CABA y GBA el plazo es de 24 a 48 horas hábiles.""",

    """MEDIOS DE PAGO — TechStore Argentina
    Aceptamos tarjetas de crédito Visa, Mastercard y American Express.
    Las compras en 12 cuotas sin interés aplican solo para productos seleccionados.
    También aceptamos transferencia bancaria con un 10% de descuento adicional.""",

    """ATENCIÓN AL CLIENTE — TechStore Argentina
    Nuestro centro de atención opera de lunes a viernes de 9 a 18 hs.
    Podés contactarnos por WhatsApp al +54 11 4000-0000 o por email a soporte@techstore.com.ar.
    Los reclamos se resuelven en un plazo máximo de 72 horas hábiles."""
]

print(f"📄 Corpus cargado: {len(documentos)} documentos")
for i, doc in enumerate(documentos):
    print(f"  Documento {i+1}: {doc.split(chr(10))[1].strip()[:50]}...")

📄 Corpus cargado: 5 documentos
  Documento 1: Los clientes pueden devolver productos electrónico...
  Documento 2: Todos los productos tienen garantía legal de 6 mes...
  Documento 3: Los envíos al interior del país tienen un plazo de...
  Documento 4: Aceptamos tarjetas de crédito Visa, Mastercard y A...
  Documento 5: Nuestro centro de atención opera de lunes a vierne...


In [11]:
# ✂️ ETAPA 1A — CHUNKING
# Por ahora usamos los documentos completos como chunks
# En el Colab 1 vamos a explorar distintas estrategias

chunks = documentos  # cada documento es un chunk
print(f"✅ Chunking simple: {len(chunks)} chunks generados")
print(f"   Tamaño promedio: {sum(len(c) for c in chunks) // len(chunks)} caracteres por chunk")

✅ Chunking simple: 5 chunks generados
   Tamaño promedio: 263 caracteres por chunk


In [12]:
# 🔢 ETAPA 1B — EMBEDDINGS + VECTOR STORE
# Usamos ChromaDB con Sentence Transformers (gratuito, sin API key)

from sentence_transformers import SentenceTransformer
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

# Cargamos el modelo de embeddings multilingüe (soporta español)
# Corre en CPU, no necesita GPU
print("📥 Cargando modelo de embeddings multilingüe...")
embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)
print(f"✅ Modelo cargado: {embedding_model.get_sentence_embedding_dimension()} dimensiones")

# Creamos una clase de embedding function compatible con ChromaDB
class STEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        return embedding_model.encode(list(input)).tolist()

# Creamos el cliente y la colección
chroma_client = chromadb.Client()

# Eliminamos la colección si ya existe (para poder re-ejecutar la celda)
try:
    chroma_client.delete_collection("techstore_politicas")
except:
    pass

coleccion = chroma_client.create_collection(
    name="techstore_politicas",
    embedding_function=STEmbeddingFunction()
)

# Indexamos los chunks
coleccion.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"✅ Vector store creado con {coleccion.count()} documentos indexados")
print(f"   Modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2")
print(f"   Métrica de similitud: coseno (por defecto en ChromaDB)")
print(f"   💰 Costo: $0 (todo corre localmente)")

📥 Cargando modelo de embeddings multilingüe...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Modelo cargado: 384 dimensiones
✅ Vector store creado con 5 documentos indexados
   Modelo de embeddings: paraphrase-multilingual-MiniLM-L12-v2
   Métrica de similitud: coseno (por defecto en ChromaDB)
   💰 Costo: $0 (todo corre localmente)


In [13]:
# 🔍 ETAPA 2A — RECUPERACIÓN
# Dada una pregunta, buscamos los chunks más relevantes por similitud semántica

def recuperar_contexto(pregunta, top_k=2):
    resultados = coleccion.query(
        query_texts=[pregunta],
        n_results=top_k
    )
    chunks_recuperados = resultados['documents'][0]
    distancias = resultados['distances'][0]
    return chunks_recuperados, distancias

pregunta = "¿Cuánto tiempo tengo para devolver un producto?"
chunks_recuperados, distancias = recuperar_contexto(pregunta, top_k=2)

print(f"❓ Pregunta: {pregunta}")
print(f"\n🔍 Chunks recuperados (top-2):")
print("=" * 60)
for i, (chunk, dist) in enumerate(zip(chunks_recuperados, distancias)):
    print(f"\n📎 Chunk {i+1} (distancia coseno: {dist:.4f}):")
    print(chunk[:200] + ("..." if len(chunk) > 200 else ""))
print("=" * 60)
print("💡 Distancia más baja = más similar a la pregunta.")
print("   La búsqueda es semántica: entiende el significado, no solo palabras.")

❓ Pregunta: ¿Cuánto tiempo tengo para devolver un producto?

🔍 Chunks recuperados (top-2):

📎 Chunk 1 (distancia coseno: 13.2481):
POLÍTICA DE DEVOLUCIONES — TechStore Argentina
    Los clientes pueden devolver productos electrónicos dentro de los 30 días corridos 
    desde la fecha de compra, siempre que presenten el ticket ori...

📎 Chunk 2 (distancia coseno: 17.9601):
POLÍTICA DE ENVÍOS — TechStore Argentina
    Los envíos al interior del país tienen un plazo de 3 a 7 días hábiles. 
    El envío es gratuito para compras superiores a $50.000. 
    Para CABA y GBA el...
💡 Distancia más baja = más similar a la pregunta.
   La búsqueda es semántica: entiende el significado, no solo palabras.


In [14]:
# 💬 ETAPA 2B — PROMPT AUMENTADO Y GENERACIÓN
# Armamos el prompt con el contexto recuperado y le preguntamos al LLM local

def rag_completo(pregunta, top_k=2, temperature=0.3, verbose=True):

    # 1. RECUPERACIÓN
    chunks_recuperados, distancias = recuperar_contexto(pregunta, top_k)
    contexto = "\n\n".join(chunks_recuperados)

    # 2. PROMPT AUMENTADO
    # Le damos instrucciones claras al modelo
    prompt_aumentado = f"""Sos un asistente de atención al cliente de TechStore Argentina.
Respondé ÚNICAMENTE basándote en la información del contexto provisto.
Si la información no está en el contexto, decí que no tenés esa información disponible.
Respondé en español, de forma clara y concisa.

CONTEXTO:
{contexto}

PREGUNTA DEL CLIENTE:
{pregunta}"""

    if verbose:
        print("📋 PROMPT COMPLETO QUE RECIBE EL LLM:")
        print("=" * 60)
        print(prompt_aumentado)
        print("=" * 60)

    # 3. GENERACIÓN CON MODELO LOCAL
    respuesta = generar_texto(
        prompt_aumentado,
        temperature=temperature,
        max_new_tokens=300
    )

    if verbose:
        print(f"\n🤖 RESPUESTA DEL SISTEMA RAG:")
        print("-" * 60)
        print(respuesta)
        print("-" * 60)

    return respuesta

# Ejecutamos el pipeline completo
rag_completo("¿Cuánto tiempo tengo para devolver un producto?")

📋 PROMPT COMPLETO QUE RECIBE EL LLM:
Sos un asistente de atención al cliente de TechStore Argentina.
Respondé ÚNICAMENTE basándote en la información del contexto provisto.
Si la información no está en el contexto, decí que no tenés esa información disponible.
Respondé en español, de forma clara y concisa.

CONTEXTO:
POLÍTICA DE DEVOLUCIONES — TechStore Argentina
    Los clientes pueden devolver productos electrónicos dentro de los 30 días corridos 
    desde la fecha de compra, siempre que presenten el ticket original y el producto 
    esté en su embalaje original sin señales de uso.

POLÍTICA DE ENVÍOS — TechStore Argentina
    Los envíos al interior del país tienen un plazo de 3 a 7 días hábiles. 
    El envío es gratuito para compras superiores a $50.000. 
    Para CABA y GBA el plazo es de 24 a 48 horas hábiles.

PREGUNTA DEL CLIENTE:
¿Cuánto tiempo tengo para devolver un producto?

🤖 RESPUESTA DEL SISTEMA RAG:
------------------------------------------------------------
Según la 

'Según la política de devoluciones proporcionada por TechStore Argentina:\n\nLos clientes pueden devolver productos electrónicos dentro de los 30 días corridos desde la fecha de compra, siempre que presenten el ticket original y el producto esté en su embalaje original sin señales de uso.'

In [15]:
# ⚡ COMPARACIÓN DIRECTA: Con RAG vs Sin RAG
# Mismo prompt, misma pregunta, con y sin contexto

pregunta = "¿Me hacen envíos gratis al interior del país?"

print("🔴 SIN RAG (LLM solo):")
print("-" * 60)
sin_rag = generar_texto(pregunta, temperature=0.3)
print(sin_rag)

print("\n🟢 CON RAG:")
print("-" * 60)
con_rag = rag_completo(pregunta, verbose=False)
print(con_rag)

print("\n" + "=" * 60)
print("💡 La diferencia:")
print("   • SIN RAG → el modelo inventa algo plausible pero potencialmente incorrecto")
print("   • CON RAG → responde con información REAL del corpus de documentos")
print("\n   🎉 Y todo esto: sin API keys, sin costos, 100% gratuito.")

🔴 SIN RAG (LLM solo):
------------------------------------------------------------
Lo siento por la confusión, pero como asistente de inteligencia artificial, no tengo acceso a información específica sobre el estado actual o cambios en los servicios de envío de empresas de transporte. Los términos y condiciones pueden variar dependiendo de la empresa y las políticas individuales que cada compañía tenga.

Para obtener la información más precisa y actualizada sobre envíos gratuitos dentro del país, te recomendaría contactar directamente con la empresa de transporte o servicio postal correspondiente. Puedes buscar su número de teléfono o correo electrónico en línea para solicitar ayuda o preguntar sobre sus políticas de envío.

🟢 CON RAG:
------------------------------------------------------------
Sí, los envíos al interior del país son gratuitos para compras superiores a $50.000.

💡 La diferencia:
   • SIN RAG → el modelo inventa algo plausible pero potencialmente incorrecto
   • CON RA

---
# 📌 Parte 4 — Más allá de RAG: el paisaje actual

RAG no es la solución definitiva. Es un punto en un espectro de enfoques que el campo sigue desarrollando.

## Las alternativas y evoluciones principales

### 1. 📄 Context Stuffing
Si la ventana de contexto del LLM es suficientemente grande, ¿para qué recuperar? Simplemente metés todo el corpus en el prompt.

- **Cuándo funciona:** corpus pequeño y acotado (un contrato, un manual corto)
- **Limitación clave:** costo, latencia, y el fenómeno *"lost in the middle"* — el modelo recuerda mejor lo que está al principio y al final del contexto

### 2. 🎯 Fine-tuning
En vez de dar contexto en el prompt, el conocimiento se "hornea" directamente en los pesos del modelo durante el entrenamiento.

- **Cuándo funciona:** conocimiento estático que no cambia frecuentemente
- **Limitación clave:** el conocimiento queda congelado. Si cambia la info, hay que reentrenar
- **Hoy se usa:** en combinación con RAG, no como reemplazo

### 3. 🕸️ GraphRAG
En vez de chunks independientes, se construye un **grafo de conocimiento** con entidades y relaciones. La recuperación navega relaciones, no solo similitud vectorial.

- **Ventaja:** mucho mejor para preguntas complejas que requieren razonamiento en múltiples pasos
- **Ejemplo:** *"¿Qué productos vendidos por el proveedor X tienen problemas de garantía relacionados con el componente Y?"*

### 4. 🤖 Agentic RAG
El LLM no sigue un pipeline fijo. Decide activamente qué buscar, cuándo buscar, si necesita más información, si reformular la query.

- **El enfoque con más tracción hoy** para aplicaciones complejas
- **Frameworks:** LangGraph, LlamaIndex Workflows

### 5. ⚡ Caching Semántico
Si alguien preguntó algo similar antes, se devuelve la respuesta cacheada sin correr todo el pipeline.

- **No es un reemplazo** sino una optimización de producción
- **Reduce costos y latencia** dramáticamente en sistemas con muchos usuarios

---

## 🗺️ El mapa completo

```
  Complejidad del sistema
        ▲
        │
  Alta  │  Fine-tuning ──→ GraphRAG ──→ Agentic RAG
        │
  Media │              RAG clásico
        │
  Baja  │  Context Stuffing
        │
        └────────────────────────────────────────────▶
                                              Dinamismo del conocimiento
                                         (estático → actualización constante)
```

> **RAG clásico es el punto de entrada.** Entenderlo bien es el prerequisito para cualquiera de las evoluciones.
>
> **Y lo mejor: todo lo que aprendiste acá funciona con herramientas gratuitas.**

---
# ✅ Resumen del Bloque 1 (Versión gratuita)

| Concepto | Lo que aprendimos |
|---|---|
| **Alucinación** | Los LLMs generan texto plausible aunque sea incorrecto |
| **Softmax + Temperature** | Convierte logits en probabilidades; temperature controla el azar |
| **Pipeline RAG** | Indexación offline + recuperación y generación en tiempo real |
| **Herramientas gratuitas** | Sentence Transformers + ChromaDB + Qwen2.5 (todo open-source) |
| **Sin API keys** | Todo corre localmente en Colab, cero costos |
| **Más allá de RAG** | Context stuffing, fine-tuning, GraphRAG, Agentic RAG |

---

> 🎉 **Felicitaciones:** completaste el Bloque 1 usando herramientas 100% gratuitas y open-source.
> Ahora ya sabés cómo funciona RAG por dentro, sin depender de APIs pagas.

## 🚀 ¿Qué sigue?

En los próximos notebooks vamos a explorar en profundidad cada decisión de diseño:

- **Colab 1:** Chunking — ¿cómo afecta el tamaño y el overlap a la calidad del retrieval?
- **Colab 2:** Embeddings — ¿qué diferencia hay entre TF-IDF y embeddings semánticos?
- **Colab 3:** Prompt Engineering — ¿cómo cambia la respuesta según cómo le pedimos al LLM?
- **Colab 4:** Modelos y Temperature — ¿qué modelo elegir y cómo afecta la temperature?